In [1]:
import pandas as pd
import numpy as np
import os

# Define path to the WELFake dataset file
dataset_path = os.path.join("..", "data", "WELFake_Dataset.csv")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
%pip install scikit-learn pandas numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\VICTUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

print("All advanced modeling packages loaded successfully!")

All advanced modeling packages loaded successfully!


In [4]:
import os
print(os.listdir("../data/"))

['.gitkeep', 'cleaned_data.csv', 'notebooks', 'WELFake_Dataset.csv']


In [5]:
import os

print("--- Current Location ---")
print(os.getcwd())

print("\n--- Files in the Data Folder ---")
try:
    print(os.listdir("../data"))
except Exception as e:
    print(f"Error accessing folder: {e}")

--- Current Location ---
c:\Users\VICTUS\Desktop\Fake_News_Detection\notebooks

--- Files in the Data Folder ---
['.gitkeep', 'cleaned_data.csv', 'notebooks', 'WELFake_Dataset.csv']


In [6]:
import os
print(os.listdir(r"C:\Users\chamo\OneDrive\Desktop\github\NLP_Group_09\data"))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\chamo\\OneDrive\\Desktop\\github\\NLP_Group_09\\data'

In [ ]:
import os

# This forces Python to look directly at your actual Windows folder path
absolute_path = r"C:\Users\chamo\OneDrive\Desktop\github\NLP_Group_09\data\WELFake_Dataset.csv"

# Load the dataset
df = pd.read_csv(absolute_path)
df = df.dropna()
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (70745, 5)


,Unnamed: 0,title,text,label,cleaned_text
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,comment expected barack obama member fyf fukyo...
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,demonstrator gathered last night exercising co...
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,dozen politically active pastor came private d...
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,r sarmat missile dubbed satan replace s fly mi...
5,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1,say one time someone sued southern poverty law...


In [ ]:
def clean_text(text):
    text = str(text).lower() # Convert to lowercase
    text = re.sub(r'\[.*?\]', '', text) # Remove text in brackets
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'<.*?>+', '', text) # Remove HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text) # Remove punctuation
    text = re.sub(r'\n', '', text) # Remove newlines
    text = re.sub(r'\w*\d\w*', '', text) # Remove words containing numbers
    return text

# Apply the cleaning to the text columns (using a sample to test it fast)
# WELFake has 'title' and 'text' columns. Let's create a combined column:
df['total_content'] = df['title'] + " " + df['text']

print("Cleaning a small sample of the dataset...")
df['cleaned_content'] = df['total_content'].head(1000).apply(clean_text)
print("Done sample cleaning!")

Cleaning a small sample of the dataset...
Done sample cleaning!


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Define the path to your team's cleaned data file
data_path = r"C:\Users\chamo\OneDrive\Desktop\github\NLP_Group_09\data\cleaned_data.csv"

# If the team hasn't generated cleaned_data.csv yet, we fall back to the original dataset
if not os.path.exists(data_path):
    data_path = r"C:\Users\chamo\OneDrive\Desktop\github\NLP_Group_09\data\WELFake_Dataset.csv"

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)

# 2. Make sure the text column exists (handle name variations)
if 'cleaned_content' not in df.columns and 'text' in df.columns:
    df['cleaned_content'] = df['text']

# 3. Securely handle missing values so TF-IDF doesn't crash
df['cleaned_content'] = df['cleaned_content'].fillna("")
df = df[df['cleaned_content'].str.strip() != ""]

# 4. Define features (X) and target labels (y)
X = df['cleaned_content']
y = df['label']

# 5. Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# 7. Fit and transform the text data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\n--- Success! ---")
print(f"Training features shape: {X_train_tfidf.shape}")
print(f"Testing features shape: {X_test_tfidf.shape}")

Loading dataset from: C:\Users\chamo\OneDrive\Desktop\github\NLP_Group_09\data\WELFake_Dataset.csv

--- Success! ---
Training features shape: (57080, 5000)
Testing features shape: (14271, 5000)


In [ ]:
# Initialize the Linear Support Vector Classifier
svm_model = LinearSVC(random_state=42, max_iter=2000)

print("Training the SVM model (this may take a few moments)...")
svm_model.fit(X_train_tfidf, y_train)
print("Model training complete!")

Training the SVM model (this may take a few moments)...
Model training complete!


In [ ]:
# Make predictions on the test set
y_pred = svm_model.predict(X_test_tfidf)

# Calculate accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"--- SVM Model Accuracy: {accuracy * 100:.2f}% ---\n")

# Print complete classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))

--- SVM Model Accuracy: 94.84% ---

Classification Report:
              precision    recall  f1-score   support

    Real (0)       0.95      0.95      0.95      6975
    Fake (1)       0.95      0.95      0.95      7296

    accuracy                           0.95     14271
   macro avg       0.95      0.95      0.95     14271
weighted avg       0.95      0.95      0.95     14271

